# Embedding inference (optional notebook)

The supported runner is the local CLI script `scripts/run_embeddings.py` via Google Colab CLI.
See the README for `colab new`, `colab drivemount`, and `colab exec`.

Embeddings are mean-pooled last hidden states from the loaded VLM, not a dedicated embedding model.

This notebook is a leftover for inspecting a session that already has Drive mounted
(for example after `colab drivemount`). It does not call `drive.mount`.

If you run it with `colab exec -f notebooks/02_embeddings_inference.ipynb`, raise `--timeout`
(model load and video inference need thousands of seconds, not the CLI default of 30).

In [ ]:
# Assumes Drive is already mounted at /content/drive (colab drivemount, or a manual mount).
REPO_DIR = "/content/drive/MyDrive/open_enricher/code"
DATA_ROOT = "/content/drive/MyDrive/open_enricher/data"
OUTPUT_DIR = "/content/drive/MyDrive/open_enricher/outputs"
VIDEO = ""

In [ ]:
import sys
from pathlib import Path

repo = Path(REPO_DIR)
assert repo.exists(), f"Repository not found: {repo}"
req_file = str(repo / "requirements.txt")
repo_str = str(repo)

%pip install -q -r {req_file}
%pip install -q -e {repo_str}

src = str(repo / "src")
if src not in sys.path:
    sys.path.insert(0, src)

In [ ]:
from open_enricher import (
    DriveConfig,
    EmbeddingConfig,
    ModelConfig,
    extract_embedding,
    list_videos,
    load_model,
    resolve_video,
    save_embedding,
)

model_config = ModelConfig(model_id="Qwen/Qwen3-VL-2B-Instruct")
drive_config = DriveConfig(root=DATA_ROOT, output_dir=OUTPUT_DIR)
embedding_config = EmbeddingConfig(num_frames=16)

videos = list_videos(drive_config.root)
print(f"Found {len(videos)} video(s) under {drive_config.root}")
for path in videos[:10]:
    print(path)

video_path = resolve_video(VIDEO, drive_config.root) if VIDEO else videos[0]
print("Using:", video_path)

In [ ]:
model, processor = load_model(model_config)
vector = extract_embedding(model, processor, video_path, embedding_config)

sidecar = {
    "video": str(video_path),
    "model_id": model_config.model_id,
    "shape": list(vector.shape),
    "dtype": str(vector.dtype),
    "num_frames": embedding_config.num_frames,
    "fps": embedding_config.fps,
}
out_path = Path(drive_config.output_dir) / "embeddings" / f"{video_path.stem}.npy"
save_embedding(out_path, vector, sidecar=sidecar)
print("Saved:", out_path)
print("Shape:", vector.shape)
vector[:8]